In [11]:
# Import required libraries
import os
import pandas as pd
from utils import log_standard_scale, peek
from pathlib import Path
RAW_DIR = Path("../data")

In [12]:
# Load the dataset
df = pd.read_csv('../data/train_transcriptomics_2020_UGA.tsv', sep='\t')
df = df.drop(columns=['transcriptomics_id'])
df.head()

,participant_id,timepoint,ensembl_gene_id,raw_count,tpm_count,material
0,2020_UGA.ID_001,0,ENSG00000000003,83,0.607222,Unknown
1,2020_UGA.ID_001,0,ENSG00000000005,0,0.000000,Unknown
2,2020_UGA.ID_001,0,ENSG00000000419,2489,68.417030,Unknown
3,2020_UGA.ID_001,0,ENSG00000000457,1221,5.885515,Unknown
4,2020_UGA.ID_001,0,ENSG00000000460,532,2.958027,Unknown


In [13]:
# Unlike 2019, raw_count has actual values in 2020; material is still all 'Unknown'
# Drop material before pivoting; keep raw_count aside — we pivot on tpm_count for consistency with 2019
print('raw_count null fraction:', df['raw_count'].isna().mean())
print('material unique values:', df['material'].unique())

raw_count null fraction: 0.0
material unique values: <ArrowStringArray>
['Unknown']
Length: 1, dtype: str


In [14]:
# Drop material (entirely 'Unknown'); drop raw_count to match 2019 pipeline (tpm_count only)
df = df.drop(columns=['raw_count', 'material'])
df.head()

,participant_id,timepoint,ensembl_gene_id,tpm_count
0,2020_UGA.ID_001,0,ENSG00000000003,0.607222
1,2020_UGA.ID_001,0,ENSG00000000005,0.000000
2,2020_UGA.ID_001,0,ENSG00000000419,68.417030
3,2020_UGA.ID_001,0,ENSG00000000457,5.885515
4,2020_UGA.ID_001,0,ENSG00000000460,2.958027


In [15]:
unique_genes_challenge = set(
    pd.read_csv(RAW_DIR / "challenge_transcriptomics.tsv", sep='\t')['ensembl_gene_id'].unique()
)
len(unique_genes_challenge)

54902

In [16]:
df['timepoint'].unique()

array([ 0, 28])

In [17]:
# Filter to only timepoints 0, 7. Timepoint 3 not in the challenge set
df = df[df['timepoint'].isin([0, 7])]
# Filter to only challenge genes
df = df[df["ensembl_gene_id"].isin(unique_genes_challenge)]

# Pivot: each row is a (participant_id, timepoint), each gene becomes a column
df_pivot = df.pivot_table(
    index='participant_id',
    columns=['timepoint', 'ensembl_gene_id'],
    values='tpm_count'
)

# Flatten multi-level columns to "TRAN2020_{gene}_d{timepoint}" format
df_pivot.columns = [f'TRAN_{gene}_d{int(tp)}' for tp, gene in df_pivot.columns]
df_pivot = df_pivot.reset_index()

print(df_pivot.shape)

(48, 32709)


In [18]:
peek(df_pivot)

,participant_id,TRAN_ENSG00000000003_d0,TRAN_ENSG00000000005_d0,TRAN_ENSG00000000419_d0,TRAN_ENSG00000000457_d0,TRAN_ENSG00000000460_d0,TRAN_ENSG00000000938_d0,TRAN_ENSG00000000971_d0,TRAN_ENSG00000001036_d0,TRAN_ENSG00000001084_d0,TRAN_ENSG00000001167_d0,TRAN_ENSG00000001460_d0,TRAN_ENSG00000001461_d0,TRAN_ENSG00000001497_d0,TRAN_ENSG00000001561_d0,TRAN_ENSG00000001617_d0,TRAN_ENSG00000001626_d0,TRAN_ENSG00000001629_d0,TRAN_ENSG00000001630_d0,TRAN_ENSG00000001631_d0
0,2020_UGA.ID_001,0.607222,0.000000,68.417030,5.885515,2.958027,445.721808,0.904403,36.064293,11.176851,22.870082,1.255226,10.679389,18.200555,13.225435,0.000000,0.0,16.295707,0.244313,2.700637
1,2020_UGA.ID_005,0.340844,0.000000,54.038282,3.512946,0.966797,297.316528,0.186410,17.828331,4.018199,8.843452,0.492956,7.487605,9.689692,7.863477,1.098143,0.0,10.340212,0.277784,1.263548
2,2020_UGA.ID_011,0.409859,0.000000,37.957544,3.339057,0.564065,214.556388,1.335459,31.934511,6.428574,12.357848,0.501706,7.799213,12.313939,7.323082,0.020819,0.0,9.272074,0.061654,0.801630
3,2020_UGA.ID_014,0.282524,0.000000,50.838886,4.819864,2.170226,303.721656,1.469293,22.305976,7.548064,19.439381,0.586031,11.580808,17.569934,11.934081,0.123262,0.0,15.608047,0.359421,2.762196
4,2020_UGA.ID_016,0.221375,0.012991,36.181566,2.503880,1.384539,131.903670,1.081206,15.568533,3.294352,7.826088,0.405478,6.629999,9.190907,5.077054,0.073676,0.0,5.711373,0.379905,0.684366


In [19]:
df_pivot = log_standard_scale(df_pivot)
peek(df_pivot)

,participant_id,TRAN_ENSG00000000003_d0,TRAN_ENSG00000000005_d0,TRAN_ENSG00000000419_d0,TRAN_ENSG00000000457_d0,TRAN_ENSG00000000460_d0,TRAN_ENSG00000000938_d0,TRAN_ENSG00000000971_d0,TRAN_ENSG00000001036_d0,TRAN_ENSG00000001084_d0,TRAN_ENSG00000001167_d0,TRAN_ENSG00000001460_d0,TRAN_ENSG00000001461_d0,TRAN_ENSG00000001497_d0,TRAN_ENSG00000001561_d0,TRAN_ENSG00000001617_d0,TRAN_ENSG00000001626_d0,TRAN_ENSG00000001629_d0,TRAN_ENSG00000001630_d0,TRAN_ENSG00000001631_d0
0,2020_UGA.ID_001,1.597492,-0.144338,0.958479,1.356092,2.068009,1.438942,-0.635335,0.972665,1.303730,1.229681,2.572755,0.954572,1.186381,1.290226,-0.679383,-0.26385,1.277674,-0.188510,1.293677
1,2020_UGA.ID_005,0.388573,-0.144338,0.424848,0.179409,-0.317530,0.652127,-1.651505,-0.348883,-0.570680,-0.539081,-0.055872,0.148802,-0.339112,0.255323,4.109891,-0.26385,0.219835,0.001375,-0.120341
2,2020_UGA.ID_011,0.723418,-0.144338,-0.369625,0.069969,-1.099082,0.018958,-0.197204,0.742161,0.258750,0.070523,-0.018634,0.239813,0.232711,0.117713,-0.546217,-0.26385,-0.028085,-1.324181,-0.776884
3,2020_UGA.ID_014,0.091898,-0.144338,0.287157,0.887772,1.310932,0.693523,-0.077550,0.067405,0.555561,0.919860,0.329518,1.142238,1.099394,1.082048,0.071835,-0.26385,1.175998,0.444404,1.341134
4,2020_UGA.ID_016,-0.234022,6.783866,-0.476900,-0.525478,0.339452,-0.923390,-0.444701,-0.598361,-0.900053,-0.756915,-0.440649,-0.120073,-0.463578,-0.570292,-0.219954,-0.26385,-1.094750,0.551390,-0.970482


In [20]:
# Save the cleaned dataset to the cleaned_data folder
os.makedirs('../cleaned_data', exist_ok=True)
df_pivot.to_parquet('../cleaned_data/transcriptomics_2020_UGA_cleaned.parquet', index=False)